In [1]:
from datasets import load_dataset
from pathlib import Path
import sys
from torch.utils.data import DataLoader

C:\Anaconda\envs\study_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("ai-enthusiasm-community/KTVIC")

In [3]:
# structure of dataset 
dataset['train'][0]

{'image_uid': '000000000001',
 'caption_uid': ['000000000001_1',
  '000000000001_2',
  '000000000001_3',
  '000000000001_4',
  '000000000001_5'],
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=695x456>,
 'caption_vi': ['ba chiếc thuyền đang di chuyển ở trên con sông',
  'có ba con thuyền đang di chuyển trên con sông',
  'trên dòng sông có ba con thuyền đang di chuyển',
  'ba con thuyền đang di chuyển bên một cánh đồng lúa',
  'ba chiếc thuyền đang chuyển động trên một con sông'],
 'segment_caption_vi': ['ba chiếc thuyền đang di_chuyển ở trên con sông',
  'có ba con thuyền đang di_chuyển trên con sông',
  'trên dòng sông có ba con thuyền đang di_chuyển',
  'ba con thuyền đang di_chuyển bên một cánh đồng lúa',
  'ba chiếc thuyền đang chuyển_động trên một con sông']}

# Tổng quan bộ DL
- DL gồm ảnh và caption (vietnamese) để huấn luyện cho bài toán Image caption
- Mỗi ảnh được lưu trữ ở kiểu DL PIL và mỗi ảnh có nhiều caption

In [4]:
len(dataset['train'])

3769

In [5]:
ROOT_DIR = Path.cwd().parent
ROOT_DIR

WindowsPath('D:/Code Module 6/VietNamese_Image_Captioning_with_CNN_LTSM')

In [6]:
SOURCE_DIR = ROOT_DIR/'source/'

In [7]:
sys.path.extend([str(ROOT_DIR), str(SOURCE_DIR)])

In [8]:
from organize_data import *

In [9]:
vicaptiondataset = VicaptioningDataSet(dataset=dataset, split='train',val_split=0.2,is_shuffle=True)

In [10]:
vicaptiondataset[0][0].shape

torch.Size([3, 224, 224])

In [11]:
len(vicaptiondataset)

15070

In [12]:
from vocabulary.vocab import Tokenizer, BuiltVocabFromIterator
from preprocessing.preprocessing_text import Preprocessing

In [13]:
processor = Preprocessing()

In [14]:
tokenizer = Tokenizer()

In [15]:
vicaptionDataLoader = DataLoader(dataset=vicaptiondataset, batch_size=512)

In [16]:
def yield_tokens(list_documentation):
    for doc in list_documentation:
        doc = processor.tranfer2Lower(processor.remove_punctuation_digit(doc))
        yield tokenizer(doc)

In [17]:
# get list caption of all data
documentations = tuple() 
for _, captions in vicaptionDataLoader:
    documentations +=captions

In [18]:
# build vocab
vocab = BuiltVocabFromIterator(
    iterator= yield_tokens(documentations),vocab_size=600,special_tokens=['<padd>', '<start>', '<end>', '<unk>']
)
del documentations
del vicaptionDataLoader

In [19]:
vocab.set_default(key='<unk>')

In [20]:
vocab.stoi()

{'<padd>': 0,
 '<start>': 1,
 '<end>': 2,
 '<unk>': 3,
 'có': 4,
 'một': 5,
 'ở': 6,
 'đang': 7,
 'người': 8,
 'hiện': 9,
 'xuất': 10,
 'trên': 11,
 'bên': 12,
 'hàng': 13,
 'của': 14,
 'trong': 15,
 'nhiều': 16,
 'chiếc': 17,
 'những': 18,
 'nữ': 19,
 'phụ': 20,
 'áo': 21,
 'trước': 22,
 'mặc': 23,
 'đường': 24,
 'màu': 25,
 'đứng': 26,
 'được': 27,
 'gái': 28,
 'xe': 29,
 'cô': 30,
 'phía': 31,
 'đàn': 32,
 'ông': 33,
 'nhà': 34,
 'khu': 35,
 'con': 36,
 'hai': 37,
 'sự': 38,
 'bức': 39,
 'cửa': 40,
 'ảnh': 41,
 'quầy': 42,
 'mặt': 43,
 'di': 44,
 'là': 45,
 'vài': 46,
 'cây': 47,
 'cái': 48,
 'tay': 49,
 'ngồi': 50,
 'máy': 51,
 'chuyển': 52,
 'bày': 53,
 'hình': 54,
 'đây': 55,
 'siêu': 56,
 'thị': 57,
 'xanh': 58,
 'ngôi': 59,
 'và': 60,
 'trắng': 61,
 'đi': 62,
 'bán': 63,
 'vào': 64,
 'phố': 65,
 'chợ': 66,
 'ăn': 67,
 'nhóm': 68,
 'chụp': 69,
 'các': 70,
 'khung': 71,
 'quả': 72,
 'kệ': 73,
 'đen': 74,
 'cảnh': 75,
 'này': 76,
 'cầm': 77,
 'nhau': 78,
 'đồ': 79,
 'trái': 80,
 '

In [21]:
CONFIGURATION_DIR = ROOT_DIR/'configuration'

In [22]:
sys.path.append(str(CONFIGURATION_DIR))

In [23]:
PATH_SAVE_VOCAB = CONFIGURATION_DIR/'vocab.json'

In [24]:
vocab.save(str(PATH_SAVE_VOCAB))

Save success vocab at D:\Code Module 6\VietNamese_Image_Captioning_with_CNN_LTSM\configuration\vocab.json


In [25]:
sentence_test = 'toi rat thích xem phim'

In [26]:
tokenizer(sentence_test)

['toi', 'rat', 'thích', 'xem', 'phim']

In [27]:
vocab(tokenizer(sentence_test))

[3, 3, 3, 403, 3]